# Gradient Descent


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Nonlinear Optimization

- The class of nonlinear optimization problems is very broad
- The main objective:
$$\begin{align}
\min_x \quad & f(x) \\
\text{s.t.}\quad & c_E(x) = 0 \\
& c_I(x) \le 0
\end{align}$$
- We'll focus on the *unconstrained problem*: $$\min_x f(x)$$

### Gradient Descent

- First order (uses first derivative -- gradient) for finding **local minimum** of a **differentiable** function $$f: X \rightarrow \mathbb{R}$$
- We assume $X \subseteq \mathbb{R}^N$

#### Reminders

- Recall: a function $f$ is minimized (or maximized!) where $\frac{df}{dx} = 0$
- Also recall that the *gradient* of $f$ is: $$\nabla f = \begin{bmatrix}\frac{\partial f}{\partial x_1} \\ \frac{\partial f}{\partial x_2} \\ \vdots \\ \frac{\partial f}{\partial x_N}\end{bmatrix}$$
- $\nabla f \big|_{x = \hat{x}}$ gives the direction of greatest *increase* in $f$, starting from the point $x = \hat{x}$

#### Intuition

- If $\nabla f$ represents direction of greatest increase in $f$...
- then moving *away* from this point will move us in a direction of greatest decrease
- The gradient descent algorithm is an iterative algorithm that builds on this insight

#### The Algorithm

The gradient descent algorithm proceeds as follows:

- Initialization:
    - Choose a starting point $x_0$,
    - convergence tolerance $\epsilon$,
    - maximum number of iterations $T$,
    - step size $\alpha$
- Iteration: for iteration $i = 0, \cdots T$
    1. Evaluate $dx_i \equiv \nabla f(x)\big|_{x=x_i}$
    2. Compute $x_{i+1} = x_i - \alpha dx_i$
    3. Check $\rho(dx_i) < \epsilon \quad \big | \quad i = T$
    
        - If True, stop and return $x_{i+1}$
        - If False, continue to next iteration

### Gradient Descent By Hand

Below we have written a very direct implementation of gradient descent

In [ ]:
def grad_descent(df, x0, epsilon=1e-3, T=200, alpha=0.1):
    """
    Given a gradient function df, staritng starting point x0,
    stopping parameters epsilon and T, and a learning rate alpha;
    find a local minimum of f(x) near x_0 via gradient descent
    """
    x = np.copy(x0)
    trace = []
    for i in range(T):
        df_i = df(x)
        xp = x - alpha * df_i
        err = max(abs(df_i))
        status = {"x": xp, "i": i, "err": err}
        trace.append(status)
        if err < epsilon:
            return trace
        x[:] = xp[:]

    return trace


To test our algorithm, let's define a function `f` for which we know the minimum

In [ ]:
def f(x):
    return -np.exp(-(x[0]**2 + x[1]**2))

def f2(x):
    return  -np.sinc(x[0]/2)-np.exp(-((x[0]-1)**2 + (x[1]-1/2)**2))

In [ ]:
def plot_surf(f, **kw):
    L, n = 2, 400
    x = np.linspace(-L, L, n)
    y = x.copy()
    X, Y = np.meshgrid(x, y)

    # set up 3d plot
    fig, ax = plt.subplots(figsize=(10, 6), subplot_kw={'projection': '3d'})

    Z = f([X, Y])

    ax.plot_surface(X, Y, Z, cmap="viridis", **kw)
    return ax


plot_surf(f);

As you can see from the above we have a global minimum at (0,0)

Let's define a function that can compute the gradient of our function: $$\nabla f(x) = \begin{bmatrix} -2 x_1 f(x) \\ -2 x_2 f(x) \end{bmatrix}$$

In [ ]:
def df(x):
    return -2*np.asarray(x)*f(x)

And now we can use our `grad_descent` function to find the minimizer for the function `f`, given its gradient `df`:

In [ ]:
trace = grad_descent(df, [2, -0.3], T=1000)
opt = trace[-1]
opt

Let's redo the surface plot, this time including a scatter chart showing the path of $x_i$ over time

In [ ]:
def get_trace_xyz(f, trace):
    xy = [i["x"] for i in trace]
    x, y = zip(*xy)
    z = f([np.array(x), np.array(y)])
    return x, y, z


def plot_path(f, trace, **kw):
    ax = plot_surf(f, **kw)
    x, y, z = get_trace_xyz(f, trace)
    ax.scatter3D(x, y, z, c="red")
    ax.plot(x, y, z, c="red")
    ax.view_init(10, 295)
    return ax

plot_path(f, trace, alpha=0.3);

Another very helpful way to view the algorithm's progress is with a contour plot

In [ ]:
def plot_contour_path(f, trace, ax=None):
    L, n = 2, 400
    x = np.linspace(-L, L, n)
    y = x.copy()
    X, Y = np.meshgrid(x, y)
    Z = f([X, Y])

    # set up plot
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 6))

    CS = ax.contour(X, Y, Z)
    ax.clabel(CS, inline=True, fontsize=10)
    
    x, y, z = get_trace_xyz(f, trace)
    ax.scatter(x, y, c=np.linspace(0.5, 1, len(x)), s=8)
    ax.set_title("Convergence in {} iterations".format(len(x)));
    return ax

ax = plot_contour_path(f, trace);

### Importance of Learning rate

- Recall the update rule for $x$: $x_{i+1} = x_i - \alpha \nabla f(x) \big|_{x=x_i}$
- The parameter $\alpha$ is called the *learning rate*
- This influences two key features of the algorithm:
    - *rate of convergence*: how many iterations it takes to go from starting position to final position
    - *stability of iterations*: whether the problem remains well defined from one iteration to the next
- In general a smaller $\alpha$ means a slower rate of convergence, but more stable algorithm

#### Small Learning rate

- Let's see what happens as we make the learning rate very small
- By default, or `grad_descent` sets $\alpha$ = 0.1
- Let's try smaller values for $\alpha$

In [ ]:
def alpha_experiment(alphas):
    N = len(alphas)
    fig, ax = plt.subplots(1, N, figsize=(N*4, 4))
    for alpha, ax in zip(alphas, ax):
        trace_alpha = grad_descent(df, [2, -0.3], alpha=alpha, T=100_000)
        plot_contour_path(f, trace_alpha, ax=ax)
    fig.tight_layout()
    return fig

In [ ]:
alpha_experiment([0.1, 0.2, 0.3]);

- Notice how we ended up at the same place (near (0,0)), but when alpha became very small it took far more iterations
- Also notice that when the slope gets steeper (between (0, 1) for example), the steps are larger
    - This is a feature of gradient descent: when gradient is large (steep), steps are bigger
    - Easiest to see on chart where alpha = 0.1

#### Impact of high learning rate

- Let's now consider the alternative where $\alpha$ is large

In [ ]:
alpha_experiment([0.1, 0.5, 0.95]);

- Notice that with $\alpha = 0.5$ convergence was very fast -- only 16 iterations
- However, when we increase alpha further to $\alpha=0.95$ we needed 64 iterations, why?
    - Learning rate was too high and caused overshoot
    - Can think of gradient descent as rolling a marble on a surface with a "push" and a "snapshot" at fixed time intervals
        - Strength of push related to size of $\alpha$. If you push too hard, you overshoot

In [ ]:
trace95 = grad_descent(df, [2.0, -0.3], alpha=0.95)
plot_path(f, trace95, alpha=0.3);

The problem can get worse if we make $\alpha$ too big:

In [ ]:
grad_descent(df, [0.2, -0.3], alpha=1)

- In this case the iterations became unstable and the marble "escaped"
- To plot the escape we need to do a few iterations "by hand"

In [ ]:
def interactive_alpha_experiment(alpha_power, T=50):
    "alpha = 10**alpha_power to make it easy to do things on log10 scale"
    x0 = np.array([2.0, -0.3])
    x = np.copy(x0)
    trace = []
    
    alpha = 10 ** alpha_power

    for i in range(T):
        df_i = df(x)
        xp = x - alpha * df_i
        err = max(abs(df_i))
        status = {"x": xp, "i": i, "err": err}
        trace.append(status)
        x[:] = xp[:]

    ax = plot_path(f, trace, alpha=0.3)
    ax.set_title(f"alpha = {alpha}")
    return ax

In [ ]:
from ipywidgets import interact, widgets
interact(
    interactive_alpha_experiment, 
    alpha_power=widgets.FloatSlider(min=-3, max=3, step=0.25, value=-1),
    T=widgets.IntSlider(min=1, max=1000, step=1, value=50)
);